# bibliotecas usadas

In [49]:
import pandas as pd

# carregamento do csv e a primeira olhada no df

In [50]:
df = pd.read_csv('C:/Arquivos Vsco/Projeto/Projeto GS 2K26/data/bruto/dados_clima_bruto.csv')
df.head()

,data_hora,temperatura,umidade,precipitacao,cidade
0,2026-05-01T00:00,NaN,NaN,NaN,Sao_Paulo
1,2026-05-01T01:00,NaN,NaN,NaN,Sao_Paulo
2,2026-05-01T02:00,NaN,NaN,NaN,Sao_Paulo
3,2026-05-01T03:00,NaN,NaN,NaN,Sao_Paulo
4,2026-05-01T04:00,NaN,NaN,NaN,Sao_Paulo


In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2736 entries, 0 to 2735
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   data_hora     2736 non-null   object 
 1   temperatura   2313 non-null   float64
 2   umidade       2313 non-null   float64
 3   precipitacao  2313 non-null   float64
 4   cidade        2736 non-null   object 
dtypes: float64(3), object(2)
memory usage: 107.0+ KB


In [52]:
df.describe()

,temperatura,umidade,precipitacao
count,2313.000000,2313.000000,2313.000000
mean,16.724730,82.881971,0.130177
std,3.809662,16.892736,0.569698
min,7.300000,29.000000,0.000000
25%,14.200000,73.000000,0.000000
50%,16.100000,90.000000,0.000000
75%,18.900000,96.000000,0.000000
max,28.800000,100.000000,9.000000


# verificando se tem valores nulos ou duplicados

In [53]:
df.isnull().sum()

data_hora         0
temperatura     423
umidade         423
precipitacao    423
cidade            0
dtype: int64

In [54]:
# Remove linhas com dados incompletos
df = df.dropna(subset=['temperatura', 'umidade', 'precipitacao'])

In [66]:
df.isnull().sum()

data_hora                0
temperatura              0
umidade                  0
precipitacao             0
cidade                   0
precipitacao_24h         0
precipitacao_48h         0
temperatura_media_24h    0
umidade_max_24h          0
hora_dia                 0
dia_semana               0
chovendo                 0
risco                    0
dtype: int64

In [55]:
df.duplicated().sum()

np.int64(0)

# tratando a variável de data_hora

In [56]:
df['data_hora'] = pd.to_datetime(df['data_hora'])

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2313 entries, 141 to 2735
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   data_hora     2313 non-null   datetime64[ns]
 1   temperatura   2313 non-null   float64       
 2   umidade       2313 non-null   float64       
 3   precipitacao  2313 non-null   float64       
 4   cidade        2313 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 108.4+ KB


# feature engineering 

In [57]:
# feature 1
df['precipitacao_24h'] = (
    df.groupby('cidade')['precipitacao'].rolling(window=24, min_periods=1).sum().reset_index(0, drop=True)
)

In [58]:
# feature 2
df['precipitacao_48h'] = (
    df.groupby('cidade')['precipitacao'].rolling(window=48, min_periods=1).sum().reset_index(0, drop=True)
)

In [59]:
# feature 3
df['temperatura_media_24h'] = (
    df.groupby('cidade')['temperatura'].rolling(window=24, min_periods=1).mean().reset_index(0, drop=True)
)

In [60]:
# feature 4
df['umidade_max_24h'] = (
    df.groupby('cidade')['umidade'].rolling(window=24, min_periods=1).max().reset_index(0, drop=True)
)

In [61]:
# feature 5 (hr do dia)
df['hora_dia'] = df['data_hora'].dt.hour

In [62]:
# feature 6
df['dia_semana'] = df['data_hora'].dt.day_of_week

In [63]:
# feature 7 (se esta chovendo agr, em binário)
df['chovendo'] = (df['precipitacao'] > 0).astype(int)

# variável alvo

In [64]:
df['risco'] = 'Baixo'

df.loc[df['precipitacao_24h'] >= 20, 'risco'] = 'Médio'

df.loc[df['precipitacao_24h'] >= 40, 'risco'] = 'Alto'

# salvando o df

In [65]:
df.to_csv('../data/dados_clima_tratados.csv')